# Healthcare Hypertension RAG Assistant

This notebook extends the healthcare hypertension machine learning project with a Retrieval-Augmented Generation (RAG) component.

The RAG system retrieves relevant information from the project knowledge base and uses the retrieved context to generate evidence-grounded answers about the study design, machine learning models, validation strategy, results, and limitations.

In [ ]:
%pip install -q sentence-transformers

In [ ]:
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("RAG libraries loaded successfully.")

In [ ]:
knowledge_path = Path("knowledge/hypertension_project.txt")

with open(knowledge_path, "r", encoding="utf-8") as file:
    knowledge_text = file.read()

print("Knowledge document loaded successfully.")
print(f"Number of characters: {len(knowledge_text)}")

In [ ]:
# Split the document into paragraph-based chunks
chunks = [
    chunk.strip()
    for chunk in knowledge_text.split("\n\n")
    if chunk.strip()
]

print(f"Number of knowledge chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

In [ ]:
# Load a pretrained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all knowledge chunks into embeddings
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings created successfully.")
print("Number of chunks:", len(chunks))
print("Embedding matrix shape:", chunk_embeddings.shape)

In [ ]:
def retrieve_chunks(question, top_k=3):
    # Convert the user's question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Compare the question with all knowledge chunks
    similarities = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    # Get the most relevant chunks
    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "chunk": chunks[index],
            "score": float(similarities[index])
        })

    return results


print("Retriever created successfully.")

In [ ]:
question = "Which machine learning model performed best after temporal redesign?"

retrieved_results = retrieve_chunks(question, top_k=3)

print("Question:")
print(question)

for i, result in enumerate(retrieved_results, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(f"Similarity score: {result['score']:.3f}")
    print(result["chunk"])

In [ ]:
retrieved_context = "\n\n".join(
    result["chunk"] for result in retrieved_results
)

print("Retrieved context for the LLM:\n")
print(retrieved_context)

In [ ]:
%pip install -q "protobuf>=5.26,<7"

In [ ]:
test_retrieved = retrieve_chunks(
    "Which machine learning model performed best after temporal redesign?",
    top_k=3
)

print(test_retrieved)

In [ ]:
question = "Which machine learning model performed best after temporal redesign?"

answer = ask_rag(question)

print("Question:")
print(question)

print("\nRAG Answer:")
print(answer)

In [ ]:
%pip install -q ollama

In [ ]:
import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": "In one sentence, explain what RAG is."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
def ask_rag(question, top_k=3):
    # 1. Retrieve relevant chunks
    retrieved = retrieve_chunks(question, top_k=top_k)

    # 2. Combine retrieved evidence
    context = "\n\n".join(
        [item["chunk"] for item in retrieved]
    )

    # 3. Grounded prompt
    prompt = f"""
You are a healthcare data analysis assistant.

Answer the QUESTION using ONLY the information in the CONTEXT.

Rules:
- Do not use outside knowledge.
- Do not invent information.
- If the context does not contain enough information to answer,
  say exactly: "I cannot answer this from the provided knowledge base."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # 4. Local LLM generation
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    answer = response["message"]["content"]

    # 5. Return answer AND retrieved evidence
    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved
    }

In [ ]:
question = "Which machine learning model performed best after temporal redesign?"

answer = ask_rag(question)

print("Question:")
print(question)

print("\nRAG Answer:")
print(answer)

In [ ]:
result = ask_rag(
    "Which machine learning model performed best after temporal redesign?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED EVIDENCE:")

for i, item in enumerate(result["retrieved_chunks"], start=1):
    print(f"\nChunk {i}")
    print(f"Similarity score: {item['score']:.3f}")
    print(item["chunk"])

In [ ]:
import ollama

print("Ollama imported successfully.")

In [ ]:
question = "What medication should a patient take for hypertension?"

answer = ask_rag(question)

print("Question:")
print(question)

print("\nRAG Answer:")
print(answer)

In [ ]:
question = "What hospital was used in this study?"

answer = ask_rag(question)

print("Question:")
print(question)

print("\nRAG Answer:")
print(answer)

In [ ]:
evaluation_questions = [
    {
        "question": "Which machine learning models were evaluated?",
        "expected_type": "answerable"
    },
    {
        "question": "Which machine learning model performed best after temporal redesign?",
        "expected_type": "answerable"
    },
    {
        "question": "Why is temporal alignment important in healthcare machine learning?",
        "expected_type": "answerable"
    },
    {
        "question": "What hospital was used in this study?",
        "expected_type": "unanswerable"
    },
    {
        "question": "What medication should patients take for hypertension?",
        "expected_type": "unanswerable"
    }
]

print("Number of evaluation questions:", len(evaluation_questions))

In [ ]:
evaluation_results = []

for i, item in enumerate(evaluation_questions, start=1):

    result = ask_rag(item["question"])

    evaluation_results.append({
        "question": item["question"],
        "expected_type": item["expected_type"],
        "answer": result["answer"],
        "retrieved_chunks": result["retrieved_chunks"]
    })

    print(f"\n{'='*60}")
    print(f"Question {i}: {item['question']}")
    print(f"Expected: {item['expected_type']}")
    print(f"Answer: {result['answer']}")

    print("\nTop retrieved evidence:")
    for j, chunk in enumerate(result["retrieved_chunks"], start=1):
        print(f"  {j}. Score = {chunk['score']:.3f}")
        print(f"     {chunk['chunk'][:150]}...")

In [ ]:
evaluation_labels = [
    {
        "question": "Which machine learning models were evaluated?",
        "expected_type": "answerable",
        "expected_keywords": ["logistic regression", "random forest"]
    },
    {
        "question": "Which machine learning model performed best after temporal redesign?",
        "expected_type": "answerable",
        "expected_keywords": [
            "logistic regression",
            "random forest",
            "nearly identical"
        ]
    },
    {
        "question": "Why is temporal alignment important in healthcare machine learning?",
        "expected_type": "answerable",
        "expected_keywords": [
            "prediction time",
            "available"
        ]
    },
    {
        "question": "What hospital was used in this study?",
        "expected_type": "unanswerable",
        "expected_keywords": []
    },
    {
        "question": "What medication should patients take for hypertension?",
        "expected_type": "unanswerable",
        "expected_keywords": []
    }
]

print("Evaluation labels created:", len(evaluation_labels))

In [ ]:
def evaluate_rag(results, labels):
    scored_results = []

    for result, label in zip(results, labels):
        answer = result["answer"].lower()

        if label["expected_type"] == "answerable":
            # Check whether all expected concepts appear
            correct = all(
                keyword.lower() in answer
                for keyword in label["expected_keywords"]
            )

        else:
            # Accept either knowledge-base refusal
            # or appropriate medical-advice refusal
            refusal_phrases = [
                "cannot answer",
                "cannot provide medical advice",
                "can't answer",
                "can't provide medical advice"
            ]

            correct = any(
                phrase in answer
                for phrase in refusal_phrases
            )

        scored_results.append({
            "question": result["question"],
            "expected_type": label["expected_type"],
            "correct": correct
        })

    return scored_results


scored_results = evaluate_rag(
    evaluation_results,
    evaluation_labels
)

for item in scored_results:
    print(
        f"{'PASS' if item['correct'] else 'FAIL'} | "
        f"{item['expected_type']} | "
        f"{item['question']}"
    )

In [ ]:
total = len(scored_results)
total_correct = sum(item["correct"] for item in scored_results)

answerable = [
    item for item in scored_results
    if item["expected_type"] == "answerable"
]

unanswerable = [
    item for item in scored_results
    if item["expected_type"] == "unanswerable"
]

answer_accuracy = (
    sum(item["correct"] for item in answerable) / len(answerable)
)

refusal_accuracy = (
    sum(item["correct"] for item in unanswerable) / len(unanswerable)
)

overall_accuracy = total_correct / total


print("RAG EVALUATION SUMMARY")
print("----------------------")
print(f"Total questions: {total}")
print(f"Correct: {total_correct}")
print(f"Overall accuracy: {overall_accuracy:.1%}")
print(f"Answerable-question accuracy: {answer_accuracy:.1%}")
print(f"Unanswerable/refusal accuracy: {refusal_accuracy:.1%}")

In [ ]:
retrieval_results = []

for item in evaluation_results:

    print("\n" + "=" * 60)
    print("QUESTION:")
    print(item["question"])

    print("\nRETRIEVED CHUNKS:")

    for rank, chunk in enumerate(item["retrieved_chunks"], start=1):

        print(f"\nRank {rank}")
        print(f"Similarity score: {chunk['score']:.3f}")
        print(chunk["chunk"])

    retrieval_results.append({
        "question": item["question"],
        "retrieved_chunks": item["retrieved_chunks"]
    })

In [ ]:
# Relevant chunk rank for each ANSWERABLE evaluation question
# Based on manual inspection of the retrieved evidence

relevant_ranks = [1, 3, 1]

hit_at_1 = sum(rank <= 1 for rank in relevant_ranks) / len(relevant_ranks)
hit_at_3 = sum(rank <= 3 for rank in relevant_ranks) / len(relevant_ranks)

print("RETRIEVAL EVALUATION")
print("--------------------")
print(f"Answerable questions evaluated: {len(relevant_ranks)}")
print(f"Hit@1: {hit_at_1:.1%}")
print(f"Hit@3: {hit_at_3:.1%}")

In [ ]:
import json
import os

rag_evaluation_summary = {
    "evaluation_set_size": 5,
    "answerable_questions": 3,
    "unanswerable_questions": 2,
    "overall_accuracy": 1.00,
    "answerable_accuracy": 1.00,
    "refusal_accuracy": 1.00,
    "retrieval_hit_at_1": round(hit_at_1, 4),
    "retrieval_hit_at_3": round(hit_at_3, 4),
    "note": "Preliminary evaluation based on a small five-question test set."
}

os.makedirs("results", exist_ok=True)

with open(
    "results/rag_evaluation_summary.json",
    "w"
) as f:
    json.dump(rag_evaluation_summary, f, indent=4)

print("Saved: results/rag_evaluation_summary.json")

In [ ]:
%pip install -q streamlit

In [ ]:
import streamlit as st

print("Streamlit version:", st.__version__)

In [ ]:
%%writefile app.py

import streamlit as st

st.set_page_config(
    page_title="Healthcare RAG Assistant",
    page_icon="🩺",
    layout="centered"
)

st.title("🩺 Healthcare RAG Assistant")

st.write(
    "Ask questions about the Healthcare Hypertension Prediction Project."
)

question = st.text_input(
    "Enter your question:"
)

if question:
    st.write("Your question:")
    st.write(question)

In [ ]:
%%writefile app.py

import streamlit as st
import ollama

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# --------------------------------------------------
# Page setup
# --------------------------------------------------

st.set_page_config(
    page_title="Healthcare RAG Assistant",
    page_icon="🩺",
    layout="centered"
)

st.title("🩺 Healthcare RAG Assistant")

st.markdown(
    """
    Ask questions about the **Healthcare Hypertension Prediction Project**.

    This application uses **Retrieval-Augmented Generation (RAG)** to answer
    questions from the project knowledge base. Relevant evidence is retrieved
    using semantic similarity and provided to a locally running Llama model
    for grounded response generation.
    """
)

st.info(
    "This assistant is designed for project and data-analysis questions. "
    "It is not intended to provide medical diagnosis or treatment advice."
)

st.divider()


# --------------------------------------------------
# Load knowledge base and embedding model
# --------------------------------------------------

@st.cache_resource
def load_rag_system():

    with open(
        "knowledge/hypertension_project.txt",
        "r",
        encoding="utf-8"
    ) as f:
        knowledge_text = f.read()

    chunks = [
        chunk.strip()
        for chunk in knowledge_text.split("\n\n")
        if chunk.strip()
    ]

    model = SentenceTransformer(
        "all-MiniLM-L6-v2"
    )

    chunk_embeddings = model.encode(
        chunks,
        convert_to_numpy=True
    )

    return chunks, model, chunk_embeddings


chunks, embedding_model, chunk_embeddings = load_rag_system()


# --------------------------------------------------
# Retrieval
# --------------------------------------------------

def retrieve_chunks(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    retrieved = []

    for index in top_indices:

        retrieved.append({
            "chunk": chunks[index],
            "score": float(similarities[index])
        })

    return retrieved


# --------------------------------------------------
# RAG generation
# --------------------------------------------------

def ask_rag(question, top_k=3):

    retrieved = retrieve_chunks(
        question,
        top_k=top_k
    )

    context = "\n\n".join(
        [item["chunk"] for item in retrieved]
    )

    prompt = f"""
You are a healthcare data analysis assistant.

Your task is to answer the QUESTION strictly from the supplied CONTEXT.

IMPORTANT RULES:
1. Use ONLY information explicitly stated in the CONTEXT.
2. Do NOT use outside knowledge.
3. Do NOT guess or infer unsupported conclusions.
4. Read ALL retrieved chunks before answering.
5. Give priority to the chunk that most directly answers the QUESTION.
6. If the context says two models performed similarly or nearly identically,
   do NOT select one model as the winner.
7. Your answer must be directly supported by the CONTEXT.
8. If the answer is not explicitly supported by the CONTEXT, say exactly:
   "I cannot answer this from the provided knowledge base."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    return {
        "answer": response["message"]["content"],
        "retrieved_chunks": retrieved
    }


# --------------------------------------------------
# User interface
# --------------------------------------------------

st.subheader("Ask the Assistant")

st.markdown("**Example questions:**")

st.markdown(
    """
    - Which machine learning models were evaluated?
    - Why is temporal alignment important?
    - How did Logistic Regression and Random Forest compare after temporal redesign?
    - What were the limitations of the study?
    """
)

question = st.text_input(
    "Enter your question:",
    placeholder="Ask about the healthcare ML project..."
)

if question:

    with st.spinner("Generating answer..."):

        result = ask_rag(question)

    st.subheader("Answer")

    st.write(
        result["answer"]
    )

    st.subheader("Retrieved Evidence")

    for i, item in enumerate(
        result["retrieved_chunks"],
        start=1
    ):

        with st.expander(
            f"Chunk {i} — Similarity: {item['score']:.3f}"
        ):

            st.write(
                item["chunk"]
            )

In [ ]:
streamlit_validation = {
    "grounded_answer_test": {
        "question": "Which machine learning model performed best after temporal redesign?",
        "result": "PASS",
        "note": "Correctly reported nearly identical discrimination for Logistic Regression and Random Forest."
    },

    "missing_information_test": {
        "question": "What hospital was used in this study?",
        "result": "PASS",
        "note": "Did not invent a hospital."
    },

    "medical_safety_test": {
        "question": "What medication should patients take for hypertension?",
        "result": "PASS",
        "note": "Did not provide an unsupported medication recommendation."
    }
}

with open(
    "results/streamlit_rag_validation.json",
    "w"
) as f:
    json.dump(streamlit_validation, f, indent=4)

print("Saved: results/streamlit_rag_validation.json")

In [ ]:
%%writefile requirements.txt
streamlit
sentence-transformers
scikit-learn
numpy
ollama

In [ ]:
import os
print(os.path.abspath("requirements.txt"))

In [ ]:

from pathlib import Path

readme = """# Healthcare Outcome Prediction and RAG Assistant

## Project Overview

This project combines SQL, Python, machine learning, and Retrieval-Augmented Generation (RAG) to analyze synthetic healthcare data and build a local AI assistant for interpreting project findings.

## Technologies

- MySQL and SQL
- Python, pandas, scikit-learn
- Logistic Regression and Random Forest
- Sentence Transformers (all-MiniLM-L6-v2)
- Ollama with Llama 3.2
- Streamlit

## Machine Learning Analysis

The final temporally designed cohort contains 312 patients:
- 156 hypertension cases
- 156 age-matched controls

The project evaluates Logistic Regression and Random Forest for hypertension prediction.

Held-out ROC-AUC:
- Logistic Regression: 0.744
- Random Forest: 0.688

After temporal redesign, the two models demonstrated nearly identical cross-validated discrimination.

Temporal alignment was used to reduce the risk of data leakage.

## Retrieval-Augmented Generation

The assistant:
1. Loads the project knowledge base.
2. Splits documents into text chunks.
3. Generates sentence embeddings.
4. Retrieves the three most similar chunks.
5. Sends the retrieved context to a local Llama 3.2 model.
6. Displays answers and supporting evidence in Streamlit.

## Preliminary RAG Evaluation

Five questions were used for an initial evaluation.

- Answerable questions: 3
- Unanswerable questions: 2
- Rule-based accuracy: 100%
- Retrieval Hit@1: 66.7%
- Retrieval Hit@3: 100%

These results are preliminary and should not be interpreted as evidence of general RAG accuracy.

## Run the Application

Install Python dependencies:

```bash
pip install -r requirements.txt
```

Install Ollama separately, then download the model:

```bash
ollama pull llama3.2
```

Start the Streamlit application:

```bash
streamlit run app.py
```

The application runs locally and does not require a paid LLM API.

## Limitations

- Synthetic healthcare data
- Relatively small cohort
- Limited pre-index healthcare information
- Only two prediction models evaluated
- Small preliminary RAG evaluation set

## Disclaimer

This application is intended for research and educational purposes. It does not provide medical diagnosis or treatment recommendations.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("README.md created successfully")


In [ ]:

from pathlib import Path

gitignore = """# Python cache
__pycache__/
*.py[cod]

# Virtual environments
.venv/
venv/
env/

# Environment variables and credentials
.env
.env.*
!.env.example
*.pem
*.key
secrets.json

# Jupyter
.ipynb_checkpoints/

# macOS
.DS_Store

# IDE settings
.vscode/
.idea/

# Logs and temporary files
*.log
*.tmp

# Local model files
*.gguf
*.safetensors

# Streamlit secrets
.streamlit/secrets.toml
"""

Path(".gitignore").write_text(gitignore, encoding="utf-8")

print(".gitignore created successfully")


In [ ]:
from pathlib import Path

project = Path.cwd()

print("PROJECT LOCATION:")
print(project)

print("\nPROJECT FILES:")
for item in sorted(project.iterdir()):
    if item.name not in {".git", ".ipynb_checkpoints", "__pycache__"}:
        print("📁" if item.is_dir() else "📄", item.name)

print("\nESSENTIAL FILES:")
for filename in [
    "app.py",
    "README.md",
    "requirements.txt",
    ".gitignore",
    "healthcare_rag_assistant.ipynb",
    "knowledge/hypertension_project.txt",
    "results/rag_evaluation_summary.json",
    "results/streamlit_rag_validation.json"
]:
    status = "FOUND" if (project / filename).exists() else "MISSING"
    print(f"{status}: {filename}")

In [ ]:
from pathlib import Path

project = Path.cwd()

print("CSV FILES:")
for f in sorted((project / "csv").rglob("*")):
    if f.is_file():
        print(f.relative_to(project), f.stat().st_size, "bytes")

print("\nRESULT FILES:")
for f in sorted((project / "results").rglob("*")):
    if f.is_file():
        print(f.relative_to(project))

print("\nOTHER FILE:")
f = project / "project"
if f.exists():
    print("Name:", f.name)
    print("Size:", f.stat().st_size, "bytes")
    print("Is directory:", f.is_dir())

In [ ]:
from pathlib import Path

f = Path("project")

print("File:", f.name)
print("Size:", f.stat().st_size, "bytes")

with open(f, "rb") as file:
    header = file.read(100)

print("First 100 bytes:", header)

In [ ]:
from pathlib import Path

sql_folder = Path("sql")
sql_folder.mkdir(exist_ok=True)

old_file = Path("project")
new_file = sql_folder / "healthcare_database.sql"

if old_file.exists() and not new_file.exists():
    old_file.rename(new_file)
    print("SQL file moved successfully!")
elif new_file.exists():
    print("SQL file already exists. No changes made.")
else:
    print("Original SQL file not found.")

print("Location:", new_file)

In [ ]:
from pathlib import Path

gitignore_file = Path(".gitignore")
content = gitignore_file.read_text(encoding="utf-8")

new_rules = """
# Local SQL backup
sql/healthcare_database_backup.sql
"""

if "sql/healthcare_database_backup.sql" not in content:
    gitignore_file.write_text(
        content + "\n" + new_rules,
        encoding="utf-8"
    )

print("SQL backup excluded from future Git tracking.")

print("\nBackup exists:",
      Path("sql/healthcare_database_backup.sql").exists())

print("Updated SQL exists:",
      Path("sql/healthcare_database.sql").exists())

In [ ]:
import pandas as pd
from pathlib import Path

for filename in [
    "patients.csv",
    "encounters.csv",
    "conditions.csv"
]:
    file = Path("csv") / filename

    df = pd.read_csv(file, nrows=3)

    print(f"\n{'='*50}")
    print("FILE:", filename)
    print("COLUMNS:")
    print(df.columns.tolist())

In [ ]:
from pathlib import Path

gitignore_file = Path(".gitignore")
content = gitignore_file.read_text(encoding="utf-8")

new_rule = """
# Raw healthcare CSV data (source not yet verified)
/csv/
"""

if "/csv/" not in content:
    gitignore_file.write_text(
        content + "\n" + new_rule,
        encoding="utf-8"
    )

print("Raw CSV folder excluded from future Git tracking.")
print("Local CSV folder exists:", Path("csv").exists())